# Pascal VOC 2007+2012 - ResNet50 FP32/QAT on Kaggle

Notebook này dùng Pascal VOC 2007+2012 trên Kaggle, convert VOC XML sang COCO JSON để tái dùng pipeline hiện tại của repo.

Protocol:

- train = VOC2007 trainval + VOC2012 trainval
- val/test = VOC2007 test

Notebook được tách rõ thành 2 phần:

- FP32 baseline
- QAT resume từ checkpoint FP32


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/EchteAI')
BRANCH = 'exp/resnet50-hawq-compiler'
REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'

if REPO.exists():
    print('Repo exists, pulling latest code...')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', 'origin', BRANCH], check=False)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'psutil', 'pyyaml'], cwd=str(REPO), check=True)

print('Repository:', REPO)


In [ ]:
from pathlib import Path

WORK_ROOT = Path('/kaggle/working/pascal_voc_resnet50_run')
VOC_COCO_ROOT = Path('/kaggle/working/pascal_voc_coco')
RUNTIME_CONFIG = Path('/kaggle/working/runtime_pascal_voc_resnet50.yaml')
FP32_EXPORT_DIR = Path('/kaggle/working/export_pascal_voc_resnet50_fp32')
QAT_EXPORT_DIR = Path('/kaggle/working/export_pascal_voc_resnet50_qat')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
VOC_COCO_ROOT.mkdir(parents=True, exist_ok=True)
FP32_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
QAT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def find_vocdevkit(root='/kaggle/input'):
    root = Path(root)
    candidates = []
    for path in root.rglob('VOCdevkit'):
        if (path / 'VOC2007').is_dir() and (path / 'VOC2012').is_dir():
            candidates.append(path)
    if not candidates:
        for path in root.rglob('*'):
            if path.is_dir() and (path / 'VOC2007').is_dir() and (path / 'VOC2012').is_dir():
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError('Không tìm thấy VOCdevkit/VOC2007/VOC2012 trong /kaggle/input')
    candidates = sorted({candidate.resolve() for candidate in candidates})
    return candidates[0]

VOC_ROOT = find_vocdevkit('/kaggle/input')
print('VOC_ROOT =', VOC_ROOT)
print('WORK_ROOT =', WORK_ROOT)


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-u', 'scripts/convert_pascal_voc_to_coco.py',
    '--voc-root', str(VOC_ROOT),
    '--output-dir', str(VOC_COCO_ROOT),
    '--train-sets', 'VOC2007:trainval', 'VOC2012:trainval',
    '--val-sets', 'VOC2007:test',
]

print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)

assert (VOC_COCO_ROOT / 'instances_train.json').exists()
assert (VOC_COCO_ROOT / 'instances_val.json').exists()
print('COCO annotations ready at:', VOC_COCO_ROOT)


In [ ]:
from pathlib import Path
import json
import yaml

base = yaml.safe_load((REPO / 'configs' / 'seadronessee_resnet50_fp32_2class_1080x1920.yaml').read_text())

base['dataset']['train_images'] = str(VOC_ROOT)
base['dataset']['train_annotations'] = str(VOC_COCO_ROOT / 'instances_train.json')
base['dataset']['val_images'] = str(VOC_ROOT)
base['dataset']['val_annotations'] = str(VOC_COCO_ROOT / 'instances_val.json')
base['dataset']['test_images'] = str(VOC_ROOT)
base['dataset']['test_annotations'] = str(VOC_COCO_ROOT / 'instances_val.json')
base['dataset']['ignore_category_ids'] = []
base['dataset']['num_classes'] = 21
base['dataset']['binary_collapse_foreground'] = False
base['dataset']['workers'] = 2

base['model']['backbone'] = 'resnet50'
base['model']['trainable_backbone_layers'] = 5
base['model']['min_size'] = 800
base['model']['train_min_sizes'] = [640, 672, 704, 736, 768, 800]
base['model']['max_size'] = 1333
base['model']['anchor_statistics_min_size'] = 800

base['training']['batch_size'] = 1
base['training']['fp32_batch_size'] = 1
base['training']['qat_batch_size'] = 1
base['training']['fp32_epochs'] = 24
base['training']['qat_epochs'] = 11
base['training']['warmup_iterations'] = 200
base['training']['print_frequency'] = 50
base['training']['epoch_benchmark_images'] = 100

base['output']['directory'] = str(WORK_ROOT)
base['output']['fp32_best'] = str(WORK_ROOT / 'fp32_best.pt')
base['output']['fp32_last'] = str(WORK_ROOT / 'fp32_last.pt')
base['output']['qat_best'] = str(WORK_ROOT / 'qat_best.pt')
base['output']['qat_last'] = str(WORK_ROOT / 'qat_last.pt')
base['output']['int8_model'] = str(WORK_ROOT / 'selective_int8.pt')
base['output']['evaluation_json'] = str(WORK_ROOT / 'evaluation.json')
base['output']['benchmark_json'] = str(WORK_ROOT / 'benchmark.json')
base['output']['epoch_benchmarks'] = str(WORK_ROOT / 'epoch_benchmarks.json')

RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False, allow_unicode=True), encoding='utf-8')

train_ann = json.loads((VOC_COCO_ROOT / 'instances_train.json').read_text())
val_ann = json.loads((VOC_COCO_ROOT / 'instances_val.json').read_text())
print('Saved runtime config:', RUNTIME_CONFIG)
print('train images:', len(train_ann['images']))
print('train annotations:', len(train_ann['annotations']))
print('val images:', len(val_ann['images']))
print('val annotations:', len(val_ann['annotations']))
print('\n=== Runtime config preview ===')
print(RUNTIME_CONFIG.read_text())


## FP32 baseline

In [ ]:
import subprocess
import sys

LIMIT = None
EPOCHS_THIS_RUN = 1
MAX_STEPS = None
SKIP_BENCHMARK = False
SKIP_VALIDATION = False
INIT_CHECKPOINT = ''
PARTIAL_INIT = False

cmd = [
    sys.executable, '-u', 'scripts/train_resnet50_fp32_baseline.py',
    '--config', str(RUNTIME_CONFIG),
    '--epochs-this-run', str(EPOCHS_THIS_RUN),
]
if LIMIT is not None:
    cmd += ['--limit', str(LIMIT)]
if MAX_STEPS is not None:
    cmd += ['--max-steps', str(MAX_STEPS)]
if SKIP_BENCHMARK:
    cmd += ['--skip-benchmark']
if SKIP_VALIDATION:
    cmd += ['--skip-validation']
if INIT_CHECKPOINT:
    cmd += ['--init-checkpoint', INIT_CHECKPOINT]
    if PARTIAL_INIT:
        cmd += ['--partial-init-checkpoint']

print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)


In [ ]:
import subprocess
import sys

LIMIT = None
EPOCHS_THIS_RUN = 1
PER_GPU_BATCH_SIZE = 1
RESUME = ''
NO_FIND_UNUSED = True

cmd = [
    sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
    'scripts/train_fp32_ddp.py',
    '--config', str(RUNTIME_CONFIG),
    '--epochs-this-run', str(EPOCHS_THIS_RUN),
    '--per-gpu-batch-size', str(PER_GPU_BATCH_SIZE),
]
if LIMIT is not None:
    cmd += ['--limit', str(LIMIT)]
if RESUME:
    cmd += ['--resume', RESUME]
if NO_FIND_UNUSED:
    cmd += ['--no-find-unused-parameters']

print('Running FP32 DDP 2GPU:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)


In [ ]:
import subprocess
import sys

PER_GPU_BATCH_SIZE = 1
EPOCHS_THIS_RUN = 1
RESUME = str(WORK_ROOT / 'fp32_last.pt')

cmd = [
    sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
    'scripts/train_fp32_ddp.py',
    '--config', str(RUNTIME_CONFIG),
    '--resume', RESUME,
    '--epochs-this-run', str(EPOCHS_THIS_RUN),
    '--per-gpu-batch-size', str(PER_GPU_BATCH_SIZE),
    '--no-find-unused-parameters',
]

print('Running FP32 resume DDP 2GPU:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)


In [ ]:
from pathlib import Path
import shutil

FP32_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for name in [
    'fp32_best.pt',
    'fp32_last.pt',
    'evaluation.json',
    'benchmark.json',
    'epoch_benchmarks.json',
]:
    src = WORK_ROOT / name
    if src.exists():
        dst = FP32_EXPORT_DIR / name
        shutil.copy2(src, dst)
        print('copied:', dst)

print('\nFP32 export dir =', FP32_EXPORT_DIR)


In [ ]:
import json
import subprocess
from pathlib import Path

KAGGLE_OWNER = 'thngvs'
FP32_DATASET_SLUG = 'pascal-voc-resnet50-fp32-checkpoints'
FP32_TITLE = 'Pascal VOC ResNet50 FP32 Checkpoints'
FP32_VERSION_NOTES = 'FP32 Pascal VOC checkpoint update after latest epoch'

metadata = {
    'title': FP32_TITLE,
    'id': f'{KAGGLE_OWNER}/{FP32_DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(FP32_EXPORT_DIR / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Upload dir:', FP32_EXPORT_DIR)
print('Dataset id:', metadata['id'])
print('Metadata:', FP32_EXPORT_DIR / 'dataset-metadata.json')

probe = subprocess.run(
    ['kaggle', 'datasets', 'status', metadata['id']],
    text=True,
    capture_output=True,
)

if probe.returncode == 0:
    cmd = ['kaggle', 'datasets', 'version', '-p', str(FP32_EXPORT_DIR), '-m', FP32_VERSION_NOTES, '--dir-mode', 'zip']
    action = 'version'
else:
    cmd = ['kaggle', 'datasets', 'create', '-p', str(FP32_EXPORT_DIR), '--dir-mode', 'zip']
    action = 'create'

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print('\n===', action.upper(), 'STDOUT ===')
print(result.stdout if result.stdout else '<empty>')
print('\n===', action.upper(), 'STDERR ===')
print(result.stderr if result.stderr else '<empty>')
assert result.returncode == 0, f'Kaggle dataset {action} failed'


## QAT

In [ ]:
import subprocess
import sys

LIMIT = None
EPOCHS_THIS_RUN = 1
MAX_STEPS = None
PER_GPU_BATCH_SIZE = 1
FP32_CHECKPOINT = str(WORK_ROOT / 'fp32_best.pt')
NO_FIND_UNUSED = True

cmd = [
    sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
    'scripts/train_qat_ddp.py',
    '--config', str(RUNTIME_CONFIG),
    '--fp32-checkpoint', FP32_CHECKPOINT,
    '--epochs-this-run', str(EPOCHS_THIS_RUN),
    '--per-gpu-batch-size', str(PER_GPU_BATCH_SIZE),
]
if LIMIT is not None:
    cmd += ['--limit', str(LIMIT)]
if MAX_STEPS is not None:
    cmd += ['--max-steps', str(MAX_STEPS)]
if NO_FIND_UNUSED:
    cmd += ['--no-find-unused-parameters']

print('Running QAT DDP 2GPU:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)


In [ ]:
import subprocess
import sys

PER_GPU_BATCH_SIZE = 1
EPOCHS_THIS_RUN = 1
FP32_CHECKPOINT = str(WORK_ROOT / 'fp32_best.pt')
RESUME = str(WORK_ROOT / 'qat_last.pt')

cmd = [
    sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
    'scripts/train_qat_ddp.py',
    '--config', str(RUNTIME_CONFIG),
    '--fp32-checkpoint', FP32_CHECKPOINT,
    '--resume', RESUME,
    '--epochs-this-run', str(EPOCHS_THIS_RUN),
    '--per-gpu-batch-size', str(PER_GPU_BATCH_SIZE),
    '--no-find-unused-parameters',
]

print('Running QAT resume DDP 2GPU:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)


In [ ]:
from pathlib import Path
import shutil

QAT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for name in [
    'fp32_best.pt',
    'fp32_last.pt',
    'qat_best.pt',
    'qat_last.pt',
    'selective_int8.pt',
    'evaluation.json',
    'benchmark.json',
    'epoch_benchmarks.json',
]:
    src = WORK_ROOT / name
    if src.exists():
        dst = QAT_EXPORT_DIR / name
        shutil.copy2(src, dst)
        print('copied:', dst)

print('\nQAT export dir =', QAT_EXPORT_DIR)
print('WORK_ROOT =', WORK_ROOT)


In [ ]:
import json
import subprocess
from pathlib import Path

KAGGLE_OWNER = 'thngvs'
QAT_DATASET_SLUG = 'pascal-voc-resnet50-qat-checkpoints'
QAT_TITLE = 'Pascal VOC ResNet50 QAT Checkpoints'
QAT_VERSION_NOTES = 'QAT Pascal VOC checkpoint update after latest epoch'

metadata = {
    'title': QAT_TITLE,
    'id': f'{KAGGLE_OWNER}/{QAT_DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(QAT_EXPORT_DIR / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Upload dir:', QAT_EXPORT_DIR)
print('Dataset id:', metadata['id'])
print('Metadata:', QAT_EXPORT_DIR / 'dataset-metadata.json')

probe = subprocess.run(
    ['kaggle', 'datasets', 'status', metadata['id']],
    text=True,
    capture_output=True,
)

if probe.returncode == 0:
    cmd = ['kaggle', 'datasets', 'version', '-p', str(QAT_EXPORT_DIR), '-m', QAT_VERSION_NOTES, '--dir-mode', 'zip']
    action = 'version'
else:
    cmd = ['kaggle', 'datasets', 'create', '-p', str(QAT_EXPORT_DIR), '--dir-mode', 'zip']
    action = 'create'

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print('\n===', action.upper(), 'STDOUT ===')
print(result.stdout if result.stdout else '<empty>')
print('\n===', action.upper(), 'STDERR ===')
print(result.stderr if result.stderr else '<empty>')
assert result.returncode == 0, f'Kaggle dataset {action} failed'
